In [75]:
import pandas as pd
import os
from datetime import datetime

# df_init = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\Automation.xlsx", sheet_name='BoP edit')
df_init = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\Automation.xlsx", sheet_name='BoP edit')

path_dict = dict()
list_1 = ['IMF BOP Annual.xlsx', 'IMF BOP Quarterly.xlsx']
# directory_path= r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
directory_path= r'C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
for filename in os.listdir(directory_path):
    if filename in ['01 Emerging Market', '02 G7 Countries']:
        continue
    file_path = os.path.join(directory_path, filename)
    list_temp = []
    for filename2 in os.listdir(file_path):
        if any(word in str(filename2) for word in list_1):            
            list_temp.append(filename2)
            path_dict[filename] = list_temp
        else:
            continue

df_path = []
for k, v in path_dict.items():
    df_path.append(os.path.join(directory_path, k, v[1]))
    print(v[1]) ## to check the file name

df_all_annual = pd.DataFrame()
for x, y in enumerate(df_path):
    df = pd.read_excel(y)
    df = df[df[df.columns[0]].isin(df_init['desc'])]
    df_all_annual = pd.concat([df_all_annual, df]) 

df_all_annual.rename(columns={df_all_annual.columns[0]: 'Type'}, inplace=True)

df_all_annual_2 = df_all_annual[df_all_annual.columns[:4].to_list() + [dt for dt in df_all_annual.columns[4:].sort_values(ascending=True).to_list() if dt >= datetime(2019, 1, 1)]]

def sg_row(df:pd.DataFrame, main:bool = True) -> pd.DataFrame:
    var = df.columns[:4] if main else df.columns[4:]
    return df[var]

sg_add = {
 'FDI Assets':{'FDI Equity Assets': 0.74,'FDI Debt Assets': 0.26},
 'FDI Liabilities': {'FDI Equity Liabilities': 0.85,'FDI Debt Liabilities': 0.15},
 'Portfolio Assets': {'Portfolio Equity Assets': 0.57, 'Portfolio Debt Assets': 0.43}, 
 'Portfolio Liabilities': {'Portfolio Equity Liabilities': 0.67, 'Portfolio Debt Liabilities': 0.33},
 'Other Investment Assets': {'Currency and Deposits Assets': 0.5, 'Loans Assets': 0.29, 'Insurance and Pension Assets': 0.02, 'Trade Credits and Advances Assets': 0.09, 'OI Others Assets': 0.1},
 'Other Investment Liabilities': {'Currency and Deposits Liabilities': 0.5, 'Loans Liabilities': 0.29, 'Insurance and Pension Liabilities': 0.02, 'Trade Credits and Advances Liabilities': 0.09, 'OI Others Liabilities': 0.1},
 }

m = df_all_annual_2[(df_all_annual_2['Region']== 'Singapore')]['Type'].to_list()
df_init[(df_init['desc'].isin(m))]['group'].unique().tolist()

sg_item = [ 'BoP: Financial Account: Direct Investment: Assets',
            'BoP: Financial Account: Direct Investment: Liabilities',
            'BoP: Financial Account: Portfolio Investment: Assets',
            'BoP: Financial Account: Portfolio Investment: Liabilities',
            'BoP: Financial Account: Other Investment: Assets',
            'BoP: Financial Account: Other Investment: Liabilities']

sg_df = pd.DataFrame(columns=df_all_annual_2.columns)

# ensure the zip between sg_item and sg_add is correct
for x, y in zip(sg_item, sg_add.values()):
    multiplier = list(y.values())
    row_type = list(y.keys())
    df_temp = df_all_annual_2[(df_all_annual_2['Type'] == x) &(df_all_annual_2['Region']== 'Singapore')]
    # print(multiplier, row_type)
    for n in range(len(row_type)):
        col1 = sg_row(df_temp, main=True)
        col1.iat[0, 0] = list(df_init[(df_init['short_title']==row_type[n])]['desc'])[0]
        col2 = sg_row(df_temp, main=False) * multiplier[n]
        merged = pd.concat([col1, col2], axis=1)
        sg_df = pd.concat([sg_df, merged], axis=0)

df_init_tw = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\Automation.xlsx", sheet_name='Taiwan BoP')
df_tw = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data\Chinese Taipei (Taiwan) (TW)\TW CB BOP Quarterly.xlsx")
df_tw.rename(columns={df_tw.columns[0]: 'Type'}, inplace=True)
df_tw = df_tw[df_tw['Type'].isin(df_init_tw['desc'])]

for x, y in enumerate(df_tw['Type'].to_list()):
    df_tw.iat[x,0] = list(df_init_tw[(df_init_tw['desc'] == y)]['desc2'])[0]

selected_columns = [col for col in df_tw.columns[4:] if pd.to_datetime(col) >= datetime(2019, 1, 1)]
final_columns = list(df_tw.columns[:4]) + selected_columns
df_tw = df_tw[final_columns]

# concat to the main df
df_all_annual_2 = pd.concat([df_all_annual_2, sg_df, df_tw], axis=0)

init_dict = df_init.to_dict()
df_dict = dict()
for x in init_dict['desc'].values():
    df_temp = df_all_annual_2[(df_all_annual_2[df_all_annual_2.columns[0]] == x)].reset_index(drop=True)
    df_dict[x] = df_temp

# calculate the sum of the columns (1H and 2H) for each year
num1, num2 = 0, 1
df_col = df_all_annual_2.columns[4:][:-1]
df_temp_col_1 = df_all_annual_2[df_all_annual_2.columns[:4]].reset_index(drop=True)

for x in range(len(df_col)//2):
    df_col_temp = (df_all_annual_2[df_col[num1]] + df_all_annual_2[df_col[num1 + 1]]).div(1000)
    year = df_col[num1].year
    df_col_temp.reset_index(drop=True, inplace=True)
    df_temp_col_1[f'{num2}H{year}'] = df_col_temp
    num1 += 2
    num2 += 1
    if num2 == 3:
        num2 = 1

df_temp_col_1.drop(columns=['Last Update Time', 'Unit'], inplace=True)

# df_all_annual_2[df_all_annual_2.columns[:8]]

df_temp_col_1


BN IMF BOP Quarterly.xlsx
KH IMF BOP Quarterly.xlsx
CH IMF BOP Quarterly.xlsx
HK IMF BOP Quarterly.xlsx
IN IMF BOP Quarterly.xlsx
ID IMF BOP Quarterly.xlsx
KR IMF BOP Quarterly.xlsx
LA IMF BOP Quarterly.xlsx
MY IMF BOP Quarterly.xlsx
MG IMF BOP Quarterly.xlsx
NP IMF BOP Quarterly.xlsx
PG IMF BOP Quarterly.xlsx
PH IMF BOP Quarterly.xlsx
SG IMF BOP Quarterly.xlsx
LK IMF BOP Quarterly.xlsx
TH IMF BOP Quarterly.xlsx
VN IMF BOP Quarterly.xlsx


C:\Users\Admin\AppData\Local\Temp\ipykernel_22216\3641161086.py:75: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sg_df = pd.concat([sg_df, merged], axis=0)


,Type,Region,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
0,BoP: Current Account,Brunei,NaN,NaN,1.519496,-1.005783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BoP: Current Account: Goods,Brunei,NaN,NaN,1.963671,-0.604498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BoP: Current Account: Services,Brunei,NaN,NaN,-0.414720,-0.440466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BoP: Current Account: Primary Income,Brunei,NaN,NaN,0.151028,0.208608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BoP: Current Account: Secondary Income,Brunei,NaN,NaN,-0.180482,-0.169427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,BoP: Financial Account: Other Investment: Liab...,Taiwan,3.508,-3.466,19.300000,-13.802000,17.090,-9.175,11.283,-13.250,8.136,3.241,-0.196,NaN,NaN
549,BoP: Financial Account: Other Investment: Liab...,Taiwan,2.351,1.240,5.835000,3.171000,-3.023,2.285,3.732,0.054,-1.183,-7.527,-1.608,NaN,NaN
550,BoP: Financial Account: Other Investment: Liab...,Taiwan,2.457,-3.189,1.244000,-0.018000,-5.619,-3.081,5.932,-1.918,2.087,-3.210,1.735,NaN,NaN
551,BoP: Financial Account: Official Reserve Assets,Taiwan,5.174,11.484,11.083000,37.259000,14.165,6.828,4.071,6.812,9.939,4.403,5.132,NaN,NaN


In [1]:
['Brunei',
'Cambodia',
'China',
'Hong Kong',
'India',
'Indonesia',
'Korea',
'Lao',
'Malaysia',
'Mongolia',
'Myanmar',
'Nepal',
'Papua New Guinea',
'Philippines',
'Singapore',
'Sri Lanka',
'Taiwan',
'Thailand',
'Vietnam',]


['Brunei',
 'Cambodia',
 'China',
 'Hong Kong',
 'India',
 'Indonesia',
 'Korea',
 'Lao',
 'Malaysia',
 'Mongolia',
 'Myanmar',
 'Nepal',
 'Papua New Guinea',
 'Philippines',
 'Singapore',
 'Sri Lanka',
 'Taiwan',
 'Thailand',
 'Vietnam']

In [66]:
#checking

temp_list = []

for x in df_init[['desc', 'short_title']].values.tolist():
    # print(len(df_dict[x]), x) if len(df_dict[x]) == 0 else print(x)
    temp_list.append([x[1], len(df_dict[x[0]])])
    # print([x[1], len(df_dict[x[0]])])

# print(temp_list)
# temp_list
# df_dict['BoP: Financial Account: Other Investment: Liabilities: Other Accounts Receivable/Payable'].head(2)
# df_init.desc.to_list()

# [16, 14, 14, 17, 13, 13, 11, 17, 7, 16, 15, 8, 13, 16, 17, 17, 17, 17, 17, 17, 17, 16, 16, 17, 14, 13, 11, 17, 6, 16, 16, 8, 14, 15, 14]
# [16, 14, 14, 17, 13, 13, 11, 17, 7, 16, 15, 8, 13, 16, 17, 17, 17, 17, 17, 17, 17, 16, 16, 17, 14, 13, 11, 17, 6, 16, 16, 8, 14, 15, 14]
# [16, 15, 15, 17, 14, 14, 11, 17, 7, 17, 16, 9, 14, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 15, 14, 11, 17, 6, 17, 17, 9, 15, 16, 14]

# ['FDI Assets', 17] -- 16
# ['FDI Equity Assets', 16] -- 16
# ['FDI Debt Assets', 16] -- 16
# ['Portfolio Assets', 18] -- 16
# ['Portfolio Equity Assets', 15] -- 16 ###############################
# ['Portfolio Debt Assets', 15] -- 16 ###############################
# ['Financial Derivative Assets', 12] -- 13 ###############################
# ['Other Investment Assets', 18] -- 16
# ['OI Equity Assets', 8] -- 7/6
# ['Currency and Deposits Assets', 18] -- 16
# ['Loans Assets', 17] -- 15
# ['Insurance and Pension Assets', 9] -- 8
# ['Trade Credits and Advances Assets', 15] -- 14
# ['OI Others Assets', 18] -- 16
# ['Official Reserve Assets', 18] -- 16
# ['Current Account', 18] -- 16
# ['Trade Balance (Goods)', 18] -- 16
# ['Trade Balance (Services)', 18] -- 16
# ['Investment Income', 18] -- 16
# ['Current Transfers', 18] -- 16
# ['FDI Liabilities', 18] -- 16
# ['FDI Equity Liabilities', 18] -- 16
# ['FDI Debt Liabilities', 18] -- 16
# ['Portfolio Liabilities', 18] -- 16
# ['Portfolio Equity Liabilities', 16] -- 16
# ['Portfolio Debt Liabilities', 15] -- 16 ###############################
# ['Financial Derivative Liabilities', 12] -- 12
# ['Other Investment Liabilities', 18] -- 16
# ['OI Equity Liabilities', 7] -- 6
# ['Currency and Deposits Liabilities', 18] -- 16
# ['Loans Liabilities', 18] -- 16
# ['Insurance and Pension Liabilities', 9] -- 7
# ['Trade Credits and Advances Liabilities', 16] -- 15
# ['OI Others Liabilities', 17] -- 15
# ['SDR Liabilities', 15] -- 14

In [72]:
# 'BoP: Financial Account: Portfolio Investment: Assets: Equity'
# 'BoP: Financial Account: Portfolio Investment: Assets: Debt Securities'
# 'BoP: Financial Account: Financial Derivatives & Employee Stock Options: Assets'
# 'BoP: Financial Account: Portfolio Investment: Liabilities: Debt Securities'

# df_dict['BoP: Financial Account: Portfolio Investment: Assets: Equity']

In [73]:
# df_all_annual_2[df_all_annual_2.columns[4:]]
# (df_all_annual_2[x] + df_all_annual_2[x+1]).div(1000)
num1, num2 = 0, 1
df_col = df_all_annual_2.columns[4:][:-1]
df_temp_col_1 = df_all_annual_2[df_all_annual_2.columns[:4]].reset_index(drop=True)

for x in range(len(df_col)//2):
    df_col_temp = (df_all_annual_2[df_col[num1]] + df_all_annual_2[df_col[num1 + 1]]).div(1000)
    year = df_col[num1].year
    df_col_temp.reset_index(drop=True, inplace=True)
    df_temp_col_1[f'{num2}H{year}'] = df_col_temp
    num1 += 2
    num2 += 1
    if num2 == 3:
        num2 = 1

df_temp_col_1.drop(columns=['Last Update Time', 'Unit'], inplace=True)
df_temp_col_1

# len((df_all_annual_2[df_col[num]] + df_all_annual_2[df_col[num + 1]]).div(1000))
# df_temp_col_1

,Type,Region,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
0,BoP: Current Account,Brunei,NaN,NaN,1.519496,-1.005783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BoP: Current Account: Goods,Brunei,NaN,NaN,1.963671,-0.604498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BoP: Current Account: Services,Brunei,NaN,NaN,-0.414720,-0.440466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BoP: Current Account: Primary Income,Brunei,NaN,NaN,0.151028,0.208608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BoP: Current Account: Secondary Income,Brunei,NaN,NaN,-0.180482,-0.169427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,BoP: Financial Account: Other Investment: Liab...,Taiwan,3.508,-3.466,19.300000,-13.802000,17.090,-9.175,11.283,-13.250,8.136,3.241,-0.196,NaN,NaN
549,BoP: Financial Account: Other Investment: Liab...,Taiwan,2.351,1.240,5.835000,3.171000,-3.023,2.285,3.732,0.054,-1.183,-7.527,-1.608,NaN,NaN
550,BoP: Financial Account: Other Investment: Liab...,Taiwan,2.457,-3.189,1.244000,-0.018000,-5.619,-3.081,5.932,-1.918,2.087,-3.210,1.735,NaN,NaN
551,BoP: Financial Account: Official Reserve Assets,Taiwan,5.174,11.484,11.083000,37.259000,14.165,6.828,4.071,6.812,9.939,4.403,5.132,NaN,NaN


In [80]:
test2 = df_all_annual_2[(df_all_annual_2['Region'] == 'Singapore')]['Type'].unique().tolist()
test = df_all_annual_2[(df_all_annual_2['Type'].isin(test2)) & (df_all_annual_2['Region'] == 'Singapore')]
for x in range(len(test)):
    test.iat[x, 0] = df_init[(df_init['desc'] == test2[x])]['short_title'].values[0]
# test.iat[0, 0] = df_init[(df_init['desc']==test2)]['short_title'].values[0]
test[test.columns[:8]]

# len(df_init)

,Type,Region,Unit,Last Update Time,2019-03-01 00:00:00,2019-06-01 00:00:00,2019-09-01 00:00:00,2019-12-01 00:00:00
0,Current Account,Singapore,USD mn,2024-09-26,14526.980143,14734.021224,16087.425469,15098.063664
3,Trade Balance (Goods),Singapore,USD mn,2024-09-26,22145.198199,25310.822045,24963.958505,23887.487164
6,Trade Balance (Services),Singapore,USD mn,2024-09-26,3525.872887,2922.245587,4561.442629,3952.545108
9,Investment Income,Singapore,USD mn,2024-09-26,-9606.333506,-11637.121620,-11584.662369,-10841.865923
12,Current Transfers,Singapore,USD mn,2024-09-26,-1537.757437,-1861.924788,-1853.313297,-1900.102684
17,FDI Assets,Singapore,USD mn,2024-09-26,13831.106518,18699.863074,20115.298851,15082.440956
18,FDI Liabilities,Singapore,USD mn,2024-09-26,22930.538127,21176.927967,28230.597702,33514.009095
20,Portfolio Assets,Singapore,USD mn,2024-09-26,8900.716026,68550.858233,22924.863057,13113.833064
21,Portfolio Liabilities,Singapore,USD mn,2024-09-26,1383.701188,4703.041714,-1597.653788,1694.880446
23,Financial Derivative Assets,Singapore,USD mn,2024-09-26,942.275042,2615.995892,3692.738378,3729.206396


In [109]:
import pandas as pd
import requests
from io import BytesIO
import concurrent.futures as cf  # MultiThreaded code processing
import time
from cache_pandas import timed_lru_cache
# from data import Data

from pathlib import Path


# path = Path(__file__).parent
# path1 = path / "test.xlsx"
# path2 = path / "financial stress index.xlsx"
# path3 = path / "section 2.xlsx"
# print(path2)


@timed_lru_cache(seconds=None, maxsize=None)
def dfs2() -> tuple[pd.DataFrame, pd.DataFrame]:
    sc2_half = pd.read_excel(r'C:\Users\Admin\OneDrive\Desktop\data science project\cfm-nlp\cfm-nlp\section 2.xlsx', sheet_name="sc2_half")
    sc2_quarter = pd.read_excel(r'C:\Users\Admin\OneDrive\Desktop\data science project\cfm-nlp\cfm-nlp\section 2.xlsx', sheet_name="sc2_quarter")
    return sc2_half, sc2_quarter

sc2_half, sc2_quarter = dfs2()

date_temp = sc2_quarter.columns[4:].to_list()
date_temp2 = [dt.strftime("%Y-%m") for dt in date_temp]
rename_date = {k: v for k, v in zip(date_temp, date_temp2)}
# list(rename_date.values())
sc2_half['Region'].unique().tolist()

['Brunei',
 'Cambodia',
 'China',
 'Hong Kong SAR (China)',
 'India',
 'Indonesia',
 'South Korea',
 'Laos',
 'Malaysia',
 'Mongolia',
 'Nepal',
 'Papua New Guinea',
 'Philippines',
 'Singapore',
 'Sri Lanka',
 'Thailand',
 'Vietnam',
 'Taiwan']

In [104]:
import pandas as pd 

df = pd.DataFrame({'A': [0, 1, 2, 3, 4],
                   'B': [5, 6, 7, 8, 9],
                   'C': ['a', 'b', 'c', 'd', 'e']})

df

,A,B,C
0,0,5,a
1,1,6,b
2,2,7,c
3,3,8,d
4,4,9,e


In [106]:
df['A'].replace([0, 1, 2, 3], [4, 3, 2, 1])

0    4
1    3
2    2
3    1
4    4
Name: A, dtype: int64

In [2]:
import pandas as pd

df_init = pd.read_excel(r"Automation.xlsx", sheet_name='BoP edit')
short_title = df_init['short_title'].to_list()
description = df_init['desc'].to_list()

for x, y in zip(short_title, description):
    print(x, y, sep='-------------')

FDI Assets-------------BoP: Financial Account: Direct Investment: Assets
FDI Equity Assets-------------BoP: Financial Account: Direct Investment: Assets: Equity
FDI Debt Assets-------------BoP: Financial Account: Direct Investment: Assets: Debt Instruments
Portfolio Assets-------------BoP: Financial Account: Portfolio Investment: Assets
Portfolio Equity Assets-------------BoP: Financial Account: Portfolio Investment: Assets: Equity
Portfolio Debt Assets-------------BoP: Financial Account: Portfolio Investment: Assets: Debt Securities
Financial Derivative Assets-------------BoP: Financial Account: Financial Derivatives & Employee Stock Options: Assets
Other Investment Assets-------------BoP: Financial Account: Other Investment: Assets
OI Equity Assets-------------BoP: Financial Account: Other Investment: Assets: Other Equity
Currency and Deposits Assets-------------BoP: Financial Account: Other Investment: Assets: Currency & Deposits
Loans Assets-------------BoP: Financial Account: Othe

In [40]:
dict1 = df_init[['group', 'short_title']].groupby("group")["short_title"].apply(list).reset_index().to_dict(orient='list')
dict2 = {k: {item: item for item in v} for k, v in zip(dict1['group'], dict1['short_title'])}

dict2

{'Current Account': {'Current Account': 'Current Account',
  'Trade Balance (Goods)': 'Trade Balance (Goods)',
  'Trade Balance (Services)': 'Trade Balance (Services)',
  'Investment Income': 'Investment Income',
  'Current Transfers': 'Current Transfers'},
 'FDI Assets': {'FDI Assets': 'FDI Assets',
  'FDI Equity Assets': 'FDI Equity Assets',
  'FDI Debt Assets': 'FDI Debt Assets'},
 'FDI Liabilities': {'FDI Liabilities': 'FDI Liabilities',
  'FDI Equity Liabilities': 'FDI Equity Liabilities',
  'FDI Debt Liabilities': 'FDI Debt Liabilities'},
 'Financial Derivative Assets': {'Financial Derivative Assets': 'Financial Derivative Assets'},
 'Financial Derivative Liabilities': {'Financial Derivative Liabilities': 'Financial Derivative Liabilities'},
 'Official Reserve Assets': {'Official Reserve Assets': 'Official Reserve Assets'},
 'Other Investment Assets': {'Other Investment Assets': 'Other Investment Assets',
  'OI Equity Assets': 'OI Equity Assets',
  'Currency and Deposits Assets':

In [10]:
from processed_data import dfs2

all_df = dfs2()

In [4]:
all_df[3]

,Group,Type,Region,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
0,Assets,Direct Investment,Asia-Pacific,149.43,180.51,180.62,185.83,225.41,243.15,273.82,238.90,224.55,231.26,241.25,0,0
1,Assets,Portfolio Equity,Asia-Pacific,69.71,90.83,90.13,203.80,220.57,114.63,100.88,79.49,83.95,96.05,184.61,0,0
2,Assets,Portfolio Debt,Asia-Pacific,141.73,66.64,53.63,35.34,75.61,81.60,128.05,122.46,125.56,65.33,193.69,0,0
3,Assets,Financial Derivatives,Asia-Pacific,-30.79,-19.37,-31.32,-28.78,-58.16,-33.89,-26.05,-67.26,-17.17,-53.29,-32.69,0,0
4,Assets,Currency & Deposits,Asia-Pacific,3.80,159.86,115.00,182.00,163.47,166.65,129.16,61.91,-50.35,158.94,17.40,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,Current Account,Goods Trade,Asia-Pacific,178.36,287.74,242.10,517.22,316.76,415.79,342.33,348.48,305.63,379.37,350.59,0,0
115,Current Account,Services Trade,Asia-Pacific,-70.25,-74.17,-48.64,-43.74,-14.06,8.44,30.99,21.57,-22.32,-37.22,-38.57,0,0
116,Current Account,Primary Income,Asia-Pacific,-50.06,-94.53,-83.74,-121.13,-99.27,-141.10,-186.08,-124.99,-124.71,-146.90,-136.91,0,0
117,Current Account,Secondary Income,Asia-Pacific,62.67,67.01,59.72,74.58,70.31,76.77,77.12,90.83,83.47,92.44,87.96,0,0


In [21]:
import plotly.express as px
import pandas as pd

dataF = all_df[3].drop(['Group','Region'], axis=1).T.reset_index()
dataF.columns = dataF.iloc[0]
dataF = dataF.iloc[1:]
dataF.rename(columns={'Type': 'Date'}, inplace=True)
dataF = dataF.melt(id_vars="Date", var_name="Type", value_name="Value")
dataF

,Date,Type,Value
0,1H2019,Direct Investment,149.43
1,2H2019,Direct Investment,180.51
2,1H2020,Direct Investment,180.62
3,2H2020,Direct Investment,185.83
4,1H2021,Direct Investment,225.41
...,...,...,...
1542,1H2023,Current Account Balance,242.06
1543,2H2023,Current Account Balance,287.69
1544,1H2024,Current Account Balance,263.08
1545,2H2024,Current Account Balance,0


In [22]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(name='A', x=['Jan', 'Feb', 'Mar'], y=[10, 20, 30]),
    go.Bar(name='B', x=['Jan', 'Feb', 'Mar'], y=[15, 25, 35])
])

fig.update_layout(
    legend=dict(
        orientation="h",      # Horizontal legend
        y=-0.2,               # Move it below the chart
        x=0.5,                # Center it horizontally
        xanchor='center',
        yanchor='top'
    ),
    barmode='group'
)

fig.show()